In [2]:
import numpy as np
import itertools
import math
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wl, wlexpr
import heapq
import pandas as pd

# session = WolframLanguageSession(kernel = "C:/Program Files/Wolfram Research/Wolfram/14.2/WolframKernel.exe")
session = WolframLanguageSession(kernel = r"C:\Program Files\Wolfram Research\Wolfram\14.2\WolframKernel.exe")

In [3]:
from wolframclient.evaluation import WolframLanguageSession
from wolframclient.language import wl, wlexpr


In [4]:
#array of multipoliers and hunters(inhabitants)
arr = np.array([[(10,1), (80,6), (37,3), (17,1), (90,10)],
                [(31,2), (50,4), (20,2), (73,4), (89,8)]])

In [5]:
#Threshold limit formula
thresh = np.zeros((2, 5))
for i in range(2):
    for j in range(5):
        mult, hunt = arr[i, j]
        thresh[i, j] = (mult - 50*hunt)/5000
with np.printoptions(precision=3, suppress=True):
    print(thresh)

[[-0.008 -0.044 -0.023 -0.007 -0.082]
 [-0.014 -0.03  -0.016 -0.025 -0.062]]


In [6]:
#MaxMin approach
def maximin1():
    """Solves the maximin optimization problem for a single expedition.

    Parameters
    ----------
    
    Returns
    -------
    argmax : list of tuple
        Maximizers.
    max_val : float
        Maximal profit.
        
    """
    max_val = float('-inf')
    argmax = []
    for mult, hunt in arr.reshape((10, 2)):
        val = (mult / (hunt + 100))*1000
        if math.isclose(val, max_val):
            argmax.append((mult, hunt))
        elif val > max_val:
            argmax = [(mult, hunt)]
            max_val = val
    return (argmax, max_val)
    ######test

In [7]:
maximin1()

([(89, 8)], 824.074074074074)

In [8]:


def maximize2():
    """Solves the maximin optimization problem for two expeditions. Returns only one solution, there may be other.

    Parameters
    ----------
    
    Returns
    -------
    argmax1 : tuple
        First expedition.
    argmax2 : tuple
        Second expedition.
    max_val : float
        Maximal profit.
        
    """
    max_val = float('-inf')
    max_mult1, max_hunt1 = None, None
    max_mult2, max_hunt2 = None, None
    for (mult1, hunt1), (mult2, hunt2) in itertools.combinations(arr.reshape((10, 2)), 2):
        val_temp = session.evaluate(wlexpr(f'NMinimize[{{{mult1}/({hunt1} + 100*p1) + {mult2}/({hunt2} + 100*p2), 0 <= p1, 0 <= p2, p1 + p2 <= 1}}, {{p1, p2}}]'))[0]
        val = -50000 + val_temp * 1000
        if math.isclose(val, max_val):
            print("collision")
        if val > max_val:
            max_mult1, max_hunt1 = mult1, hunt1
            max_mult2, max_hunt2 = mult2, hunt2
            max_val = val
    return ((max_mult1, max_hunt1), (max_mult2, max_hunt2), max_val)

In [9]:
#natural prior
ratios = np.zeros((2, 5))
for i in range(2):
    for j in range(5):
        mult, hunt = arr[i, j]
        ratios[i, j] = mult/hunt
with np.printoptions(precision=2, suppress=True):
    print("Ratios:", ratios, sep='\n')

Ratios:
[[10.   13.33 12.33 17.    9.  ]
 [15.5  12.5  10.   18.25 11.12]]


In [10]:
#normalize
shares = ratios / np.sum(ratios)
with np.printoptions(precision=3, suppress=True):
    print("Natural prior:", shares, sep='\n')

Natural prior:
[[0.077 0.103 0.096 0.132 0.07 ]
 [0.12  0.097 0.077 0.141 0.086]]


In [11]:
ratios = np.zeros((2, 5))
for i in range(2):
    for j in range(5):
        mult, hunt = arr[i, j]
        ratios[i, j] = mult/hunt
ratios = ratios**5
shares = ratios / np.sum(ratios)
with np.printoptions(precision=3, suppress=True):
    print("Natural prior with exponent 5:", shares, sep='\n')

Natural prior with exponent 5:
[[0.017 0.073 0.049 0.246 0.01 ]
 [0.155 0.053 0.017 0.35  0.029]]


In [12]:
def fee(n):
    """Compute the fee for a total of n expeditions.

    Parameters
    ----------
    n : int
        Number of expeditions.
    
    Returns
    -------
    float
        Fee.
    """
    if n == 1:
        return 0
    if n == 2:
        return -50000

def payoff(mults, hunts, shares):
    """Compute the final profit after the expeditions.

    Parameters
    ----------
    mults : list of int
        Multipliers for each destination.
    hunts : list of int
        Hunters for each destination.
    shares : list of int
        Shares for each destination.
    
    Returns
    -------
    float
        Profit.
    """
    return 1000 * sum([mult/(hunt + 100*share) for (mult, hunt, share) in zip(mults, hunts, shares)]) + fee(len(mults))

def maximize_prior_top(shares, k):
    """Given the prior, compute solutions that yield top k profits.

    Parameters
    ----------
    shares : list of int
        Shares for each destination.
    k : int
        Number of solutions
    
    Returns
    -------
    list of tuple
        Top k profits and optimal expeditions.
    """
    datas = [(mult, hunt, share) for ((mult, hunt), share) in zip(arr.reshape((10, 2)), shares.reshape(10))]
    heap = []
    iterables = [itertools.combinations(datas, n_exp) for n_exp in range(1, 4)]
    for (i, data) in enumerate(itertools.chain.from_iterable(iterables)):
        mults = [tupl[0] for tupl in data]
        hunts = [tupl[1] for tupl in data]
        shares = [tupl[-1] for tupl in data]
        val = payoff(mults, hunts, shares)
        expeditions = list(zip(mults, hunts))
        if i < k:
            heapq.heappush(heap, (val, expeditions))
        elif val > heap[0][0]:
            heapq.heappop(heap)
            heapq.heappush(heap, (val, expeditions))
    return sorted(heap, reverse=True)

In [13]:

for i in range(10):
    shares = ratios**i
    shares = shares / np.sum(shares)
    res = maximize_prior_top(shares, 1)
    print("Exponent:", i, "Profit:", f"{res[0][0]:.2f}", "Optimal expeditions:", res[0][1])

TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'

In [16]:
session = WolframLanguageSession(kernel = r"C:\Program Files\Wolfram Research\Wolfram\14.2\WolframKernel.exe")
session.evaluate(wl.WolframAlpha("number of moons of Saturn", "Result"))

Quantity[145, IndependentUnit['moons']]